In [1]:
import duckdb

from config import (
    DB_PATH,
    JOINED_PARQUET,
    SENSORY_DIR,
    SENSORY_SUBSET_PARQUET,
)

In [2]:
APPAREL_TERMS = [
    "shirt", "t-shirt", "tee", "top", "blouse", "tank",
    "hoodie", "sweatshirt", "sweater", "jacket", "coat",
    "dress", "skirt",
    "pant", "pants", "legging", "leggings", "jogger", "joggers",
    "shorts",
    "sock", "socks", "stocking", "stockings",
    "underwear", "brief", "briefs", "bra",
    "pajama", "pajamas", "sleepwear",
    "slipper", "slippers", "shoe", "shoes",
    "glove", "gloves",
    "leotard",
]

SENSORY_PRODUCT_TERMS = [
    "sensory",
    "sensory-friendly",
    "adaptive",
    "autism",
    "autistic",
    "seamless",
    "tagless",
    "compression",
    "weighted",
    "soft",
    "comfort",
    "comfortable",
    "sensitive skin",
    "orthopedic",
    "cushion",
    "cushioned",
]

PRODUCT_SEARCH_COLUMNS = [
    "product_title",
    "features",
    "description",
    "details",
]
EXCLUDE_TERMS = [
    "polishing cloth",
    "watch",
    "keychain",
    "jewelry",
    "ring",
    "necklace",
    "bracelet",
    "earring",
    "pendant",
    "belly button",
    "body piercing",
    "dog",
    "pet",
    "cleaning",
    "polishing",
    "bluetooth",
    "headset",
    "speaker",
]

In [3]:
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def make_like_condition(columns, terms):
    conditions = []

    for col in columns:
        for term in terms:
            escaped = sql_escape(term.lower())
            conditions.append(
                f"LOWER(COALESCE(CAST({col} AS VARCHAR), '')) LIKE '%{escaped}%'"
            )

    return " OR\n            ".join(conditions)

In [4]:
def main():
    SENSORY_DIR.mkdir(parents=True, exist_ok=True)

    print("Connecting to DuckDB in memory...")

    con = duckdb.connect()

    print("\nLoading joined_reviews from Parquet...")

    con.execute(
        """
        CREATE OR REPLACE TABLE joined_reviews AS
        SELECT *
        FROM read_parquet(?)
        """,
        [str(JOINED_PARQUET)],
    )

    apparel_condition = make_like_condition(
        ["product_title"],
        APPAREL_TERMS
    )

    sensory_condition = make_like_condition(
        PRODUCT_SEARCH_COLUMNS,
        SENSORY_PRODUCT_TERMS
    )

    exclude_condition = make_like_condition(
        ["product_title", "features", "description", "details"],
        EXCLUDE_TERMS
    )

    print("\nCreating cleaner sensory apparel subset...")

    con.execute(
        f"""
        CREATE OR REPLACE TABLE sensory_reviews AS
        SELECT *
        FROM joined_reviews
        WHERE
            (
                {apparel_condition}
            )
            AND
            (
                {sensory_condition}
            )
            AND NOT
            (
                {exclude_condition}
            )
        """
    )

    summary = con.execute(
        """
        SELECT
            COUNT(*) AS sensory_review_rows,
            COUNT(DISTINCT parent_asin) AS sensory_products,
            COUNT(DISTINCT asin) AS sensory_asins,
            COUNT(DISTINCT user_id) AS sensory_users,
            ROUND(AVG(rating), 2) AS avg_rating
        FROM sensory_reviews
        """
    ).fetchdf()

    print("\nSensory subset summary:")
    print(summary)

    print("\nSaving sensory subset Parquet...")

    output_path = str(SENSORY_SUBSET_PARQUET).replace("\\", "/")

    con.execute(
        f"""
        COPY sensory_reviews
        TO '{output_path}'
        (FORMAT PARQUET)
        """
    )

    print("\nSaved sensory subset to:")
    print(SENSORY_SUBSET_PARQUET)

    con.close()

In [5]:
if __name__ == "__main__":
    main()

Connecting to DuckDB in memory...

Loading joined_reviews from Parquet...

Creating cleaner sensory apparel subset...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sensory subset summary:
   sensory_review_rows  sensory_products  sensory_asins  sensory_users  \
0               135609             35376          42966         130737   

   avg_rating  
0         4.0  

Saving sensory subset Parquet...

Saved sensory subset to:
/Users/IvyGuo/Documents/GitHub/AIL-sensory-fair/data/processed/sensory/Amazon_Fashion_sensory_subset.parquet
